# GALVAT Trial 8: clean merge of the outsourced worker dataset

This notebook performs one additive merge from immutable inputs:

- Base checkpoint: `galvat_trial8_roboflow_appended_2026-07-22_15-09-09.zip`
- Worker source: Roboflow project `15-priority-classes`, version 1
- Worker delivery contract: 7,238 images and 16,110 annotations

The workflow restores the base checkpoint into a new timestamped directory, audits and charts all 44 classes, validates the complete worker export before writing, excludes the three known poor-quality empty-label images, appends the remaining 7,235 annotated worker images with all mapped labels, proves that every base image and label file is unchanged, charts the merged counts, and saves a new checkpoint ZIP to Drive.

There are no automated image-quality filters, per-class caps, or duplicate-image exclusions. The only exclusions are the three manually reviewed empty-label images. Exact duplicates are appended because this recovery notebook prioritizes preserving the complete delivered annotation set; duplicate relationships are recorded for later leakage review.


In [ ]:
# Cell 1: Setup
!pip -q install pyyaml pandas pillow tqdm matplotlib seaborn roboflow

from google.colab import drive
drive.mount('/content/drive')

import datetime
import hashlib
import json
import math
import re
import shutil
import zipfile
from collections import Counter
from getpass import getpass
from pathlib import Path, PurePosixPath

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml
from roboflow import Roboflow
from tqdm.auto import tqdm

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLITS = ('train', 'valid', 'test')
STAMP = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')


In [ ]:
# Cell 2: Immutable input and isolated output paths

SOURCE_ZIP = Path(
    '/content/drive/MyDrive/galvat_trial8_dataset_checkpoints/'
    'galvat_trial8_roboflow_appended_2026-07-22_15-09-09.zip'
)
DRIVE_OUT = Path('/content/drive/MyDrive/galvat_trial8_dataset_checkpoints')

WORK_ROOT = Path(f'/content/galvat_trial8_worker_merge_{STAMP}')
RESTORE_ROOT = WORK_ROOT / 'restore'
WORK_DS = WORK_ROOT / 'dataset'
WORK_META = WORK_ROOT / 'metadata'
WORKER_SRC_ROOT = WORK_ROOT / 'worker_source' / 'fifteen_rf_01'

assert SOURCE_ZIP.exists(), f'Missing source checkpoint: {SOURCE_ZIP}'
assert not WORK_ROOT.exists(), (
    f'Run directory already exists: {WORK_ROOT}. Restart the runtime for a clean rerun.'
)

for path in [RESTORE_ROOT, WORK_META, WORKER_SRC_ROOT.parent, DRIVE_OUT]:
    path.mkdir(parents=True, exist_ok=True)

print('Immutable source ZIP:', SOURCE_ZIP)
print('Isolated work root:', WORK_ROOT)


In [ ]:
# Cell 3: Restore only the current dataset tree from the selected checkpoint

def extract_primary_dataset(zip_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(zip_path, 'r') as archive:
        yaml_roots = [
            PurePosixPath(member.filename).parent
            for member in archive.infolist()
            if PurePosixPath(member.filename).name == 'data.yaml'
            and PurePosixPath(member.filename).parent.name == 'dataset'
        ]
        assert yaml_roots, f'No dataset/data.yaml found inside {zip_path}'

        non_restore_roots = [root for root in yaml_roots if 'restore' not in root.parts]
        candidate_roots = non_restore_roots or yaml_roots
        dataset_root_in_zip = min(candidate_roots, key=lambda root: len(root.parts))
        dataset_prefix = str(dataset_root_in_zip).rstrip('/') + '/'

        selected_members = [
            member for member in archive.infolist()
            if member.filename == str(dataset_root_in_zip)
            or member.filename.startswith(dataset_prefix)
        ]
        assert selected_members, f'No files found under {dataset_root_in_zip}'

        for member in selected_members:
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError(f'Unsafe ZIP member path: {member.filename}')
            archive.extract(member, destination)

    print('Selected dataset tree inside ZIP:', dataset_root_in_zip)
    print('Ignored nested historical dataset trees:', len(yaml_roots) - 1)
    return destination / Path(*dataset_root_in_zip.parts)


print('Extracting base checkpoint...')
BASE_DS = extract_primary_dataset(SOURCE_ZIP, RESTORE_ROOT)
assert (BASE_DS / 'train' / 'images').exists(), (
    f'Selected dataset has no train/images directory: {BASE_DS}'
)
shutil.copytree(BASE_DS, WORK_DS)

DATA_YAML = WORK_DS / 'data.yaml'
assert DATA_YAML.exists(), f'Missing restored data.yaml: {DATA_YAML}'

print('Restored dataset root:', BASE_DS)
print('Fresh working dataset:', WORK_DS)


In [ ]:
# Cell 4: Reusable audit, hashing, and chart helpers

def load_yolo_names(data_yaml_path):
    with open(data_yaml_path, 'r') as handle:
        data = yaml.safe_load(handle)
    names = data.get('names')
    if isinstance(names, dict):
        indexed = {int(key): value for key, value in names.items()}
        names = [indexed[index] for index in range(len(indexed))]
    if not isinstance(names, list) or not names:
        raise ValueError(f'Invalid class names in {data_yaml_path}')
    return data, list(names)


def norm_name(value):
    return re.sub(r'[^a-z0-9]+', ' ', str(value).lower()).strip()


def sha1_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha1()
    with open(path, 'rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def label_path_for_image(image_path):
    return image_path.parent.parent / 'labels' / f'{image_path.stem}.txt'


def image_paths_for_dataset(dataset_root):
    paths = []
    for split in SPLITS:
        image_dir = dataset_root / split / 'images'
        if image_dir.exists():
            paths.extend(
                path for path in image_dir.iterdir()
                if path.is_file() and path.suffix.lower() in IMAGE_EXTS
            )
    return sorted(paths)


def audit_dataset(dataset_root, names):
    split_rows = []
    class_image_counts = Counter()
    class_box_counts = Counter()
    bad_rows = []

    for split in SPLITS:
        image_dir = dataset_root / split / 'images'
        images = sorted(
            path for path in image_dir.iterdir()
            if path.is_file() and path.suffix.lower() in IMAGE_EXTS
        ) if image_dir.exists() else []
        empty_images = 0

        for image_path in images:
            label_path = label_path_for_image(image_path)
            lines = label_path.read_text().splitlines() if label_path.exists() else []
            seen_classes = set()
            valid_rows = 0

            if not label_path.exists():
                bad_rows.append({
                    'split': split,
                    'file': image_path.name,
                    'line': 0,
                    'reason': 'missing_label_file',
                    'raw': '',
                })

            for line_number, line in enumerate(lines, start=1):
                if not line.strip():
                    continue
                parts = line.split()
                if len(parts) != 5:
                    bad_rows.append({
                        'split': split, 'file': image_path.name,
                        'line': line_number, 'reason': 'bad_column_count', 'raw': line,
                    })
                    continue
                try:
                    class_id = int(float(parts[0]))
                    x, y, width, height = [float(value) for value in parts[1:]]
                except Exception:
                    bad_rows.append({
                        'split': split, 'file': image_path.name,
                        'line': line_number, 'reason': 'parse_error', 'raw': line,
                    })
                    continue
                if class_id < 0 or class_id >= len(names):
                    bad_rows.append({
                        'split': split, 'file': image_path.name,
                        'line': line_number, 'reason': 'bad_class_id', 'raw': line,
                    })
                    continue
                values = [x, y, width, height]
                if not all(math.isfinite(value) for value in values):
                    bad_rows.append({
                        'split': split, 'file': image_path.name,
                        'line': line_number, 'reason': 'nonfinite_coord', 'raw': line,
                    })
                    continue
                if not (
                    0 <= x <= 1 and 0 <= y <= 1
                    and 0 < width <= 1 and 0 < height <= 1
                ):
                    bad_rows.append({
                        'split': split, 'file': image_path.name,
                        'line': line_number, 'reason': 'bad_coords', 'raw': line,
                    })
                    continue

                class_name = names[class_id]
                class_box_counts[(split, class_name)] += 1
                seen_classes.add(class_name)
                valid_rows += 1

            if valid_rows == 0:
                empty_images += 1
            for class_name in seen_classes:
                class_image_counts[(split, class_name)] += 1

        split_rows.append({
            'split': split,
            'images': len(images),
            'empty_or_invalid_label_images': empty_images,
        })

    class_rows = []
    for class_name in names:
        row = {'class': class_name}
        for split in SPLITS:
            row[f'{split}_images'] = class_image_counts[(split, class_name)]
            row[f'{split}_boxes'] = class_box_counts[(split, class_name)]
        row['total_images'] = sum(row[f'{split}_images'] for split in SPLITS)
        row['total_boxes'] = sum(row[f'{split}_boxes'] for split in SPLITS)
        class_rows.append(row)

    return (
        pd.DataFrame(split_rows),
        pd.DataFrame(class_rows),
        pd.DataFrame(bad_rows),
    )


def chart_class_counts(class_df, title, output_path, color):
    plot_df = class_df.sort_values('total_images', ascending=True)
    plt.figure(figsize=(12, max(9, len(plot_df) * 0.27)))
    sns.barplot(data=plot_df, y='class', x='total_images', color=color)
    plt.title(title)
    plt.xlabel('Images containing class')
    plt.ylabel('')
    plt.tight_layout()
    plt.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.show()


def find_data_yaml(source_dir):
    candidates = sorted(
        Path(source_dir).rglob('data.yaml'),
        key=lambda path: len(path.parts),
    )
    if not candidates:
        raise FileNotFoundError(f'No data.yaml found under {source_dir}')
    return candidates[0]


In [ ]:
# Cell 5: Audit, chart, and fingerprint the untouched base checkpoint

data_yaml, NAMES = load_yolo_names(DATA_YAML)
name_to_id = {name: index for index, name in enumerate(NAMES)}

EXPECTED_BASE_NAMES = [
    'tupperware', 'lemon', 'jug', 'canned_good', 'fork', 'knife', 'spoon',
    'drinking_glass', 'wine_glass', 'coffee_mug', 'salt_pepper_shaker',
    'measuring_cup', 'medicine_bottle', 'pot', 'pan', 'cellphone', 'wallet',
    'purse', 'cane', 'keys', 'toilet', 'urinal', 'paper_towel_dispenser',
    'soap_dispenser', 'trash_can', 'light_switch', 'electrical_outlet',
    'railing', 'tv_remote', 'writing_device', 'eyeglasses', 'hearing_aid',
    'toothbrush', 'toothpaste', 'deodorant', 'razor', 'stairs', 'exit_sign',
    'bathroom_sign', 'thermostat', 'door', 'window', 'chair', 'table',
]
assert NAMES == EXPECTED_BASE_NAMES, (
    'The July 20 ontology or class-ID order does not match the expected Trial 8 '
    f'ontology. Found: {NAMES}'
)

data_yaml['path'] = str(WORK_DS)
data_yaml['train'] = 'train/images'
data_yaml['val'] = 'valid/images'
data_yaml['test'] = 'test/images'
data_yaml['names'] = {index: name for index, name in enumerate(NAMES)}
data_yaml['nc'] = len(NAMES)
with open(DATA_YAML, 'w') as handle:
    yaml.safe_dump(data_yaml, handle, sort_keys=False)

before_split_df, before_class_df, before_bad_df = audit_dataset(WORK_DS, NAMES)
assert before_bad_df.empty, (
    f'Base checkpoint has {len(before_bad_df)} invalid label rows; merge stopped.'
)

display(before_split_df)
display(before_class_df.sort_values('total_images'))

PROTECTED_CLASSES = ['electrical_outlet', 'toothbrush', 'toothpaste', 'wallet']
missing_protected = [name for name in PROTECTED_CLASSES if name not in name_to_id]
assert not missing_protected, f'Protected classes missing from ontology: {missing_protected}'

protected_before = before_class_df.set_index('class')['total_images'].to_dict()
empty_protected = {
    class_name: protected_before.get(class_name, 0)
    for class_name in PROTECTED_CLASSES
    if protected_before.get(class_name, 0) <= 0
}
assert not empty_protected, (
    f'The selected base ZIP has no images for protected classes: {empty_protected}'
)

print('Protected class counts before merge:')
display(before_class_df[before_class_df['class'].isin(PROTECTED_CLASSES)])

before_split_df.to_csv(WORK_META / 'before_split_counts.csv', index=False)
before_class_df.to_csv(WORK_META / 'before_class_counts.csv', index=False)

chart_class_counts(
    before_class_df,
    'GALVAT Trial 8 before worker merge',
    WORK_META / 'before_class_counts.png',
    'steelblue',
)

base_snapshot_rows = []
base_images = image_paths_for_dataset(WORK_DS)
for image_path in tqdm(base_images, desc='Fingerprinting immutable base files'):
    label_path = label_path_for_image(image_path)
    base_snapshot_rows.append({
        'image_relative_path': str(image_path.relative_to(WORK_DS)),
        'image_sha1': sha1_file(image_path),
        'label_relative_path': str(label_path.relative_to(WORK_DS)),
        'label_sha1': sha1_file(label_path),
    })

base_snapshot_df = pd.DataFrame(base_snapshot_rows)
base_snapshot_df.to_csv(WORK_META / 'base_file_snapshot.csv', index=False)

print('Base images:', len(base_snapshot_df))
print('Unique base image hashes:', base_snapshot_df['image_sha1'].nunique())
print('Exact duplicate base files:', len(base_snapshot_df) - base_snapshot_df['image_sha1'].nunique())


In [ ]:
# Cell 6: Download the outsourced worker's complete Roboflow export

WORKER_SOURCE = {
    'source_name': 'fifteen_rf_01',
    'workspace': 'wazir-ali-haideri',
    'project': '15-priority-classes',
    'version': 1,
}

existing_worker_yamls = list(WORKER_SRC_ROOT.rglob('data.yaml'))
if existing_worker_yamls:
    print('Worker export already downloaded:', WORKER_SRC_ROOT)
else:
    api_key = getpass('Roboflow API key: ')
    rf = Roboflow(api_key=api_key)
    project = rf.workspace(WORKER_SOURCE['workspace']).project(WORKER_SOURCE['project'])
    version = project.version(int(WORKER_SOURCE['version']))
    version.download(model_format='yolo26', location=str(WORKER_SRC_ROOT))

WORKER_DATA_YAML = find_data_yaml(WORKER_SRC_ROOT)
print('Worker data.yaml:', WORKER_DATA_YAML)


In [ ]:
# Cell 7: Strictly reconcile the worker export with the delivery report

EXPECTED_WORKER_IMAGES = 7238
EXPECTED_WORKER_BOXES = 16110
EXPECTED_EMPTY_LABEL_EXCLUSIONS = 3
EXPECTED_PRIORITY_IMAGES = {
    'railing': 517,
    'hearing_aid': 775,
    'bathroom_sign': 508,
    'tupperware': 515,
    'salt_pepper_shaker': 502,
    'wine_glass': 515,
    'drinking_glass': 530,
    'jug': 501,
    'measuring_cup': 531,
    'stairs': 544,
    'thermostat': 551,
    'paper_towel_dispenser': 534,
    'trash_can': 537,
    'light_switch': 543,
    'pan': 714,
}

_, worker_names = load_yolo_names(WORKER_DATA_YAML)
base_name_by_norm = {norm_name(name): name for name in NAMES}
special_source_targets = {
    'cell phone': 'cellphone',
    'drinkingglass': 'drinking_glass',
    'electrical outlets': 'electrical_outlet',
    'eyeglass': 'eyeglasses',
    'tvremote': 'tv_remote',
}

worker_id_to_target = {}
unmapped_worker_classes = []
for worker_id, worker_name in enumerate(worker_names):
    normalized = norm_name(worker_name)
    target_name = special_source_targets.get(normalized)
    if target_name is None and normalized == 'stairs':
        target_name = 'stairs' if 'stairs' in name_to_id else 'step_curb'
    if target_name is None:
        target_name = base_name_by_norm.get(normalized)
    if target_name not in name_to_id:
        unmapped_worker_classes.append(worker_name)
    else:
        worker_id_to_target[worker_id] = target_name

assert not unmapped_worker_classes, (
    f'Worker classes cannot be mapped safely: {unmapped_worker_classes}'
)

print('Strict source mapping:')
display(pd.DataFrame([
    {
        'worker_id': worker_id,
        'worker_class': worker_names[worker_id],
        'base_id': name_to_id[target_name],
        'base_class': target_name,
    }
    for worker_id, target_name in worker_id_to_target.items()
]))

worker_split_counts = Counter()
worker_class_image_counts = Counter()
worker_class_box_counts = Counter()
worker_hash_counts = Counter()
worker_issue_rows = []
worker_image_rows = []
worker_box_total = 0

for split in SPLITS:
    image_dir = WORKER_DATA_YAML.parent / split / 'images'
    images = sorted(
        path for path in image_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTS
    ) if image_dir.exists() else []
    worker_split_counts[split] = len(images)

    for image_path in tqdm(images, desc=f'Validating worker {split}'):
        label_path = label_path_for_image(image_path)
        image_sha1 = sha1_file(image_path)
        worker_hash_counts[image_sha1] += 1
        classes_in_image = set()
        mapped_lines = []

        if not label_path.exists():
            worker_issue_rows.append({
                'split': split, 'file': image_path.name,
                'line': 0, 'reason': 'missing_label_file', 'raw': '',
            })
            continue

        lines = [line for line in label_path.read_text().splitlines() if line.strip()]
        if not lines:
            worker_issue_rows.append({
                'split': split, 'file': image_path.name,
                'line': 0, 'reason': 'empty_label_file', 'raw': '',
            })
            continue

        for line_number, line in enumerate(lines, start=1):
            parts = line.split()
            reason = None
            if len(parts) != 5:
                reason = 'bad_column_count'
            else:
                try:
                    worker_class_id = int(float(parts[0]))
                    x, y, width, height = [float(value) for value in parts[1:]]
                except Exception:
                    reason = 'parse_error'
                else:
                    values = [x, y, width, height]
                    if worker_class_id not in worker_id_to_target:
                        reason = 'bad_or_unmapped_class_id'
                    elif not all(math.isfinite(value) for value in values):
                        reason = 'nonfinite_coord'
                    elif not (
                        0 <= x <= 1 and 0 <= y <= 1
                        and 0 < width <= 1 and 0 < height <= 1
                    ):
                        reason = 'bad_coords'

            if reason:
                worker_issue_rows.append({
                    'split': split, 'file': image_path.name,
                    'line': line_number, 'reason': reason, 'raw': line,
                })
                continue

            target_name = worker_id_to_target[worker_class_id]
            target_id = name_to_id[target_name]
            mapped_lines.append(
                f'{target_id} {x:.6f} {y:.6f} {width:.6f} {height:.6f}'
            )
            worker_box_total += 1
            worker_class_box_counts[target_name] += 1
            classes_in_image.add(target_name)

        if not mapped_lines:
            worker_issue_rows.append({
                'split': split, 'file': image_path.name,
                'line': 0, 'reason': 'no_valid_mapped_labels', 'raw': '',
            })
            continue

        for target_name in classes_in_image:
            worker_class_image_counts[target_name] += 1

        worker_image_rows.append({
            'split': split,
            'image_path': str(image_path),
            'label_path': str(label_path),
            'image_sha1': image_sha1,
            'mapped_classes': ','.join(sorted(classes_in_image)),
            'mapped_box_count': len(mapped_lines),
        })

worker_issue_df = pd.DataFrame(worker_issue_rows)
worker_image_manifest_df = pd.DataFrame(worker_image_rows)
worker_report_comparison_df = pd.DataFrame([
    {
        'class': class_name,
        'report_images': expected_count,
        'export_images': worker_class_image_counts[class_name],
        'difference': worker_class_image_counts[class_name] - expected_count,
        'matches_report': worker_class_image_counts[class_name] == expected_count,
    }
    for class_name, expected_count in EXPECTED_PRIORITY_IMAGES.items()
])

worker_total_images = sum(worker_split_counts.values())
worker_exact_duplicate_files = worker_total_images - len(worker_hash_counts)

print('Worker split counts:', dict(worker_split_counts))
print('Worker images:', worker_total_images, '/', EXPECTED_WORKER_IMAGES)
print('Worker valid boxes:', worker_box_total, '/', EXPECTED_WORKER_BOXES)
print('Exact duplicate files inside worker export:', worker_exact_duplicate_files)
print('Worker validation issues:', len(worker_issue_df))
display(worker_report_comparison_df)

empty_label_exclusions_df = worker_issue_df[
    worker_issue_df['reason'] == 'empty_label_file'
].copy()
blocking_worker_issues_df = worker_issue_df[
    worker_issue_df['reason'] != 'empty_label_file'
].copy()
WORKER_IMAGES_TO_APPEND = len(worker_image_manifest_df)

print('Manually approved empty-label exclusions:', len(empty_label_exclusions_df))
display(empty_label_exclusions_df)

worker_image_manifest_df.to_csv(WORK_META / 'worker_download_manifest.csv', index=False)
worker_issue_df.to_csv(WORK_META / 'worker_validation_issues.csv', index=False)
empty_label_exclusions_df.to_csv(
    WORK_META / 'worker_excluded_empty_label_images.csv', index=False
)
worker_report_comparison_df.to_csv(
    WORK_META / 'worker_report_reconciliation.csv', index=False
)

assert worker_total_images == EXPECTED_WORKER_IMAGES, (
    f'Worker export has {worker_total_images} images, expected {EXPECTED_WORKER_IMAGES}.'
)
assert worker_box_total == EXPECTED_WORKER_BOXES, (
    f'Worker export has {worker_box_total} valid boxes, expected {EXPECTED_WORKER_BOXES}.'
)
assert len(empty_label_exclusions_df) == EXPECTED_EMPTY_LABEL_EXCLUSIONS, (
    f'Expected {EXPECTED_EMPTY_LABEL_EXCLUSIONS} approved empty-label exclusions, '
    f'found {len(empty_label_exclusions_df)}.'
)
assert blocking_worker_issues_df.empty, (
    f'Worker export has {len(blocking_worker_issues_df)} blocking label issues.'
)
assert worker_report_comparison_df['matches_report'].all(), (
    'At least one priority-class image count differs from the delivery report.'
)

WORKER_PREFLIGHT_PASSED = True
print('WORKER PREFLIGHT PASSED')
print('Delivered images reconciled:', EXPECTED_WORKER_IMAGES)
print('Poor-quality empty-label images excluded:', EXPECTED_EMPTY_LABEL_EXCLUSIONS)
print('Annotated images ready to append:', WORKER_IMAGES_TO_APPEND)


In [ ]:
# Cell 8: Additively append all 7,235 annotated worker images and all labels

assert WORKER_PREFLIGHT_PASSED, 'Run and pass Cell 7 before merging.'

base_hashes = set(base_snapshot_df['image_sha1'])
seen_worker_hashes = set()
merge_rows = []
appended_count = 0
duplicate_base_count = 0
duplicate_worker_count = 0

for source_index, row in tqdm(
    worker_image_manifest_df.iterrows(),
    total=len(worker_image_manifest_df),
    desc='Appending worker images',
):
    image_path = Path(row['image_path'])
    label_path = Path(row['label_path'])
    image_sha1 = row['image_sha1']
    split = row['split']

    duplicate_flags = []
    if image_sha1 in base_hashes:
        duplicate_flags.append('duplicate_of_base')
        duplicate_base_count += 1
    if image_sha1 in seen_worker_hashes:
        duplicate_flags.append('duplicate_within_worker')
        duplicate_worker_count += 1
    duplicate_status = (
        'appended_' + '_and_'.join(duplicate_flags)
        if duplicate_flags
        else 'appended_unique'
    )

    mapped_lines = []
    for line in label_path.read_text().splitlines():
        if not line.strip():
            continue
        parts = line.split()
        worker_class_id = int(float(parts[0]))
        x, y, width, height = [float(value) for value in parts[1:]]
        target_name = worker_id_to_target[worker_class_id]
        mapped_lines.append(
            f'{name_to_id[target_name]} {x:.6f} {y:.6f} {width:.6f} {height:.6f}'
        )

    output_stem = f'worker15_{source_index:05d}_{image_sha1[:20]}'
    output_suffix = image_path.suffix.lower()
    output_image = WORK_DS / split / 'images' / f'{output_stem}{output_suffix}'
    output_label = WORK_DS / split / 'labels' / f'{output_stem}.txt'

    assert not output_image.exists(), f'Unexpected output collision: {output_image}'
    assert not output_label.exists(), f'Unexpected output collision: {output_label}'

    temporary_image = output_image.with_name(output_image.name + '.part')
    temporary_label = output_label.with_name(output_label.name + '.part')
    try:
        shutil.copy2(image_path, temporary_image)
        temporary_label.write_text('\n'.join(mapped_lines) + '\n')
        temporary_image.replace(output_image)
        temporary_label.replace(output_label)
    except BaseException:
        temporary_image.unlink(missing_ok=True)
        temporary_label.unlink(missing_ok=True)
        output_image.unlink(missing_ok=True)
        output_label.unlink(missing_ok=True)
        raise

    seen_worker_hashes.add(image_sha1)
    appended_count += 1
    merge_rows.append({
        'source_index': source_index,
        'source_file': image_path.name,
        'source_split': split,
        'image_sha1': image_sha1,
        'status': duplicate_status,
        'output_file': str(output_image.relative_to(WORK_DS)),
        'mapped_classes': row['mapped_classes'],
        'mapped_box_count': row['mapped_box_count'],
    })

merge_manifest_df = pd.DataFrame(merge_rows)
merge_manifest_df.to_csv(WORK_META / 'worker_merge_manifest.csv', index=False)

assert len(merge_manifest_df) == WORKER_IMAGES_TO_APPEND
assert appended_count == WORKER_IMAGES_TO_APPEND

print('Downloaded worker images:', EXPECTED_WORKER_IMAGES)
print('Excluded poor-quality empty-label images:', EXPECTED_EMPTY_LABEL_EXCLUSIONS)
print('Appended worker images:', appended_count)
print('Appended files duplicating base images:', duplicate_base_count)
print('Appended repeated files inside worker export:', duplicate_worker_count)
print('All duplicate relationships are retained in worker_merge_manifest.csv.')


In [ ]:
# Cell 9: Prove base preservation, audit the merge, and chart before vs. after

preservation_errors = []
for row in tqdm(
    base_snapshot_df.itertuples(index=False),
    total=len(base_snapshot_df),
    desc='Verifying every original base file',
):
    image_path = WORK_DS / row.image_relative_path
    label_path = WORK_DS / row.label_relative_path
    if not image_path.exists():
        preservation_errors.append({
            'path': row.image_relative_path, 'reason': 'base_image_missing'
        })
    elif sha1_file(image_path) != row.image_sha1:
        preservation_errors.append({
            'path': row.image_relative_path, 'reason': 'base_image_changed'
        })
    if not label_path.exists():
        preservation_errors.append({
            'path': row.label_relative_path, 'reason': 'base_label_missing'
        })
    elif sha1_file(label_path) != row.label_sha1:
        preservation_errors.append({
            'path': row.label_relative_path, 'reason': 'base_label_changed'
        })

preservation_errors_df = pd.DataFrame(preservation_errors)
preservation_errors_df.to_csv(
    WORK_META / 'base_preservation_errors.csv', index=False
)
assert preservation_errors_df.empty, (
    f'{len(preservation_errors_df)} original base files were lost or changed.'
)

after_split_df, after_class_df, after_bad_df = audit_dataset(WORK_DS, NAMES)
assert after_bad_df.empty, f'Merged dataset has {len(after_bad_df)} invalid label rows.'

comparison_df = before_class_df[['class', 'total_images', 'total_boxes']].merge(
    after_class_df[['class', 'total_images', 'total_boxes']],
    on='class',
    suffixes=('_before', '_after'),
)
comparison_df['image_delta'] = (
    comparison_df['total_images_after'] - comparison_df['total_images_before']
)
comparison_df['box_delta'] = (
    comparison_df['total_boxes_after'] - comparison_df['total_boxes_before']
)
comparison_df['worker_source_images'] = comparison_df['class'].map(
    lambda class_name: worker_class_image_counts[class_name]
)
comparison_df['worker_source_boxes'] = comparison_df['class'].map(
    lambda class_name: worker_class_box_counts[class_name]
)
comparison_df['worker_contribution_matches'] = (
    (comparison_df['image_delta'] == comparison_df['worker_source_images'])
    & (comparison_df['box_delta'] == comparison_df['worker_source_boxes'])
)

decreased = comparison_df[
    (comparison_df['image_delta'] < 0) | (comparison_df['box_delta'] < 0)
]
assert decreased.empty, f'Class counts decreased unexpectedly:\n{decreased}'

contribution_mismatches = comparison_df[
    ~comparison_df['worker_contribution_matches']
]
assert contribution_mismatches.empty, (
    'One or more worker classes did not append completely:\n'
    f'{contribution_mismatches}'
)

expected_final_images = int(before_split_df['images'].sum()) + appended_count
actual_final_images = int(after_split_df['images'].sum())
assert actual_final_images == expected_final_images, (
    f'Expected {expected_final_images} merged images, found {actual_final_images}.'
)

display(after_split_df)
display(comparison_df.sort_values('image_delta', ascending=False))

print('Protected classes before and after:')
display(comparison_df[comparison_df['class'].isin(PROTECTED_CLASSES)])

before_total = int(before_split_df['images'].sum())
print('Base images preserved:', before_total)
print('Worker images downloaded:', EXPECTED_WORKER_IMAGES)
print('Worker images appended:', appended_count)
print('Final merged images:', actual_final_images)
print('Bad merged label rows:', len(after_bad_df))

after_split_df.to_csv(WORK_META / 'after_split_counts.csv', index=False)
after_class_df.to_csv(WORK_META / 'after_class_counts.csv', index=False)
comparison_df.to_csv(WORK_META / 'before_after_class_comparison.csv', index=False)
after_bad_df.to_csv(WORK_META / 'after_bad_labels.csv', index=False)

chart_class_counts(
    after_class_df,
    'GALVAT Trial 8 after outsourced worker merge',
    WORK_META / 'after_class_counts.png',
    'seagreen',
)

delta_plot = comparison_df.sort_values('image_delta', ascending=True)
plt.figure(figsize=(12, max(9, len(delta_plot) * 0.27)))
sns.barplot(data=delta_plot, y='class', x='image_delta', color='darkorange')
plt.title('Worker merge contribution by class')
plt.xlabel('Additional images containing class')
plt.ylabel('')
plt.tight_layout()
plt.savefig(WORK_META / 'worker_class_image_deltas.png', dpi=180, bbox_inches='tight')
plt.show()

box_delta_plot = comparison_df.sort_values('box_delta', ascending=True)
plt.figure(figsize=(12, max(9, len(box_delta_plot) * 0.27)))
sns.barplot(data=box_delta_plot, y='class', x='box_delta', color='mediumpurple')
plt.title('Worker merge contribution by class: bounding boxes')
plt.xlabel('Additional bounding boxes')
plt.ylabel('')
plt.tight_layout()
plt.savefig(WORK_META / 'worker_class_box_deltas.png', dpi=180, bbox_inches='tight')
plt.show()

MERGE_VALIDATED = True
print('MERGE VALIDATED: every base file is unchanged and no class count decreased.')


In [ ]:
# Cell 10: Package only the validated dataset and audit metadata, then save to Drive

assert MERGE_VALIDATED, 'Cell 9 must pass before packaging.'

ZIP_LOCAL = Path(f'/content/galvat_trial8_worker_merged_{STAMP}.zip')
ZIP_DRIVE = DRIVE_OUT / ZIP_LOCAL.name
INTERNAL_ROOT = Path('galvat_trial8_worker_merged')

checkpoint_summary = {
    'created_at': STAMP,
    'source_checkpoint': str(SOURCE_ZIP),
    'worker_source': WORKER_SOURCE,
    'worker_report_expected_images': EXPECTED_WORKER_IMAGES,
    'worker_report_expected_boxes': EXPECTED_WORKER_BOXES,
    'base_images': int(before_split_df['images'].sum()),
    'worker_images_downloaded': EXPECTED_WORKER_IMAGES,
    'worker_empty_label_images_excluded': EXPECTED_EMPTY_LABEL_EXCLUSIONS,
    'worker_images_appended': int(appended_count),
    'worker_duplicates_of_base': int(duplicate_base_count),
    'worker_duplicates_within_export': int(duplicate_worker_count),
    'final_images': int(after_split_df['images'].sum()),
    'final_boxes': int(after_class_df['total_boxes'].sum()),
    'class_count': len(NAMES),
    'base_files_changed_or_missing': int(len(preservation_errors_df)),
    'bad_final_label_rows': int(len(after_bad_df)),
    'merge_policy': (
        'Reconcile all 7,238 delivered files; exclude three manually reviewed '
        'poor-quality empty-label images; append all 7,235 annotated images and mapped '
        'labels, including exact duplicates; retain every base image and label '
        'byte-for-byte; record duplicate relationships for later leakage review.'
    ),
}
(WORK_META / 'checkpoint_summary.json').write_text(
    json.dumps(checkpoint_summary, indent=2)
)

assert not ZIP_LOCAL.exists(), f'Refusing to overwrite local ZIP: {ZIP_LOCAL}'
assert not ZIP_DRIVE.exists(), f'Refusing to overwrite Drive ZIP: {ZIP_DRIVE}'

files_to_zip = []
partial_files = list(WORK_DS.rglob('*.part')) + list(WORK_META.rglob('*.part'))
assert not partial_files, f'Partial files remain from an interrupted write: {partial_files}'

dataset_suffixes = IMAGE_EXTS | {'.txt', '.yaml'}
metadata_suffixes = {'.csv', '.json', '.png'}
for path in WORK_DS.rglob('*'):
    if path.is_file() and path.suffix.lower() in dataset_suffixes:
        archive_path = INTERNAL_ROOT / 'dataset' / path.relative_to(WORK_DS)
        files_to_zip.append((path, archive_path))
for path in WORK_META.rglob('*'):
    if path.is_file() and path.suffix.lower() in metadata_suffixes:
        archive_path = INTERNAL_ROOT / 'metadata' / path.relative_to(WORK_META)
        files_to_zip.append((path, archive_path))

with zipfile.ZipFile(ZIP_LOCAL, 'w', allowZip64=True) as archive:
    for path, archive_path in tqdm(files_to_zip, desc='Creating validated checkpoint ZIP'):
        compression = (
            zipfile.ZIP_STORED
            if path.suffix.lower() in IMAGE_EXTS
            else zipfile.ZIP_DEFLATED
        )
        archive.write(path, archive_path, compress_type=compression)

with zipfile.ZipFile(ZIP_LOCAL, 'r') as archive:
    bad_member = archive.testzip()
assert bad_member is None, f'ZIP integrity failure at: {bad_member}'

print('Local ZIP:', ZIP_LOCAL)
print('ZIP size GB:', ZIP_LOCAL.stat().st_size / 1e9)
print('Copying validated checkpoint to:', ZIP_DRIVE)
shutil.copy2(ZIP_LOCAL, ZIP_DRIVE)

assert ZIP_DRIVE.exists()
assert ZIP_DRIVE.stat().st_size == ZIP_LOCAL.stat().st_size

print('Saved:', ZIP_DRIVE)
print(json.dumps(checkpoint_summary, indent=2))
